In [1]:
# ============================================================
# Task 4: Classification with Logistic Regression
# Breast Cancer Wisconsin Dataset
# ElevateLabs AI/ML Internship
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    accuracy_score, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150


In [2]:
# ============================================================
# STEP 1 — Load & Explore
# ============================================================
print("=" * 60)
print("STEP 1: Load & Explore Dataset")
print("=" * 60)

df = pd.read_csv("data.csv")

# Drop 'id' and any unnamed/empty columns
df.drop(columns=["id"], inplace=True, errors="ignore")
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print(f"\n📌 Shape: {df.shape}")
print(f"\n📌 Target Distribution:")
print(df["diagnosis"].value_counts().rename({"M": "Malignant (M)", "B": "Benign (B)"}))
print(f"\n📌 Missing Values: {df.isnull().sum().sum()}")
print(f"\n📌 Feature columns: {df.shape[1]-1}")


STEP 1: Load & Explore Dataset

📌 Shape: (569, 31)

📌 Target Distribution:
diagnosis
Benign (B)       357
Malignant (M)    212
Name: count, dtype: int64

📌 Missing Values: 0

📌 Feature columns: 30


In [3]:
# ============================================================
# STEP 2 — Preprocessing
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Preprocessing")
print("=" * 60)

# Encode target: M=1 (Malignant), B=0 (Benign)
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
print("✅ Encoded: Malignant(M)=1, Benign(B)=0")

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"✅ Features standardized with StandardScaler")


STEP 2: Preprocessing
✅ Encoded: Malignant(M)=1, Benign(B)=0
✅ Train size: 455 | Test size: 114
✅ Features standardized with StandardScaler


In [4]:
# ============================================================
# STEP 3 — Train Logistic Regression
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Train Logistic Regression Model")
print("=" * 60)

model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train_scaled, y_train)
print("✅ Model trained successfully")

# Predictions at default threshold (0.5)
y_pred      = model.predict(X_test_scaled)
y_prob      = model.predict_proba(X_test_scaled)[:, 1]  # probability of Malignant


STEP 3: Train Logistic Regression Model
✅ Model trained successfully


In [5]:
# ============================================================
# STEP 4 — Evaluation Metrics
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Model Evaluation (Default Threshold = 0.5)")
print("=" * 60)

acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
report  = classification_report(y_test, y_pred, target_names=["Benign", "Malignant"])

print(f"\n  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print(f"\n📌 Classification Report:\n{report}")

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
f1        = 2 * precision * recall / (precision + recall)

print(f"  True Negatives  (TN): {tn}  — Correctly identified as Benign")
print(f"  False Positives (FP): {fp}  — Benign misclassified as Malignant")
print(f"  False Negatives (FN): {fn}  — Malignant MISSED (most critical!)")
print(f"  True Positives  (TP): {tp}  — Correctly identified as Malignant")


STEP 4: Model Evaluation (Default Threshold = 0.5)

  Accuracy  : 0.9649  (96.49%)
  ROC-AUC   : 0.9960

📌 Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

  True Negatives  (TN): 71  — Correctly identified as Benign
  False Positives (FP): 1  — Benign misclassified as Malignant
  False Negatives (FN): 3  — Malignant MISSED (most critical!)
  True Positives  (TP): 39  — Correctly identified as Malignant


In [6]:
# ============================================================
# STEP 5 — Threshold Tuning
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Threshold Tuning")
print("=" * 60)

thresholds   = np.arange(0.1, 0.9, 0.05)
thresh_stats = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec_t  = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f1_t   = 2*prec_t*rec_t / (prec_t+rec_t) if (prec_t+rec_t) > 0 else 0
    acc_t  = accuracy_score(y_test, y_pred_t)
    thresh_stats.append({
        "Threshold": round(t, 2), "Accuracy": round(acc_t, 4),
        "Precision": round(prec_t, 4), "Recall": round(rec_t, 4),
        "F1": round(f1_t, 4), "FN": fn_t
    })
thresh_df = pd.DataFrame(thresh_stats)
print("\n📌 Threshold Analysis (lower threshold → catches more Malignant cases):")
print(thresh_df.to_string(index=False))

# Best threshold by F1
best_row = thresh_df.loc[thresh_df["F1"].idxmax()]
print(f"\n✅ Best threshold by F1: {best_row['Threshold']} → F1={best_row['F1']}")

# In medical context, prefer lower threshold (higher recall, fewer FN)
medical_thresh = 0.3
y_pred_medical = (y_prob >= medical_thresh).astype(int)
cm_med = confusion_matrix(y_test, y_pred_medical)
print(f"\n📌 Medical threshold (0.3) — minimizing missed cancer cases:")
tn_m, fp_m, fn_m, tp_m = cm_med.ravel()
print(f"  FN at 0.5 threshold: {fn}  | FN at 0.3 threshold: {fn_m}")
print(f"  Recall at 0.3: {tp_m/(tp_m+fn_m):.4f}")


STEP 5: Threshold Tuning

📌 Threshold Analysis (lower threshold → catches more Malignant cases):
 Threshold  Accuracy  Precision  Recall     F1  FN
      0.10    0.9474     0.8913  0.9762 0.9318   1
      0.15    0.9561     0.9111  0.9762 0.9425   1
      0.20    0.9561     0.9111  0.9762 0.9425   1
      0.25    0.9825     0.9762  0.9762 0.9762   1
      0.30    0.9825     0.9762  0.9762 0.9762   1
      0.35    0.9737     0.9756  0.9524 0.9639   2
      0.40    0.9737     0.9756  0.9524 0.9639   2
      0.45    0.9737     0.9756  0.9524 0.9639   2
      0.50    0.9649     0.9750  0.9286 0.9512   3
      0.55    0.9737     1.0000  0.9286 0.9630   3
      0.60    0.9649     1.0000  0.9048 0.9500   4
      0.65    0.9649     1.0000  0.9048 0.9500   4
      0.70    0.9649     1.0000  0.9048 0.9500   4
      0.75    0.9649     1.0000  0.9048 0.9500   4
      0.80    0.9474     1.0000  0.8571 0.9231   6
      0.85    0.9474     1.0000  0.8571 0.9231   6

✅ Best threshold by F1: 0.25 → F1=

In [7]:
# ============================================================
# STEP 6 — Sigmoid Function Plot
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Generating All Plots")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid curve
z = np.linspace(-10, 10, 300)
sigmoid = 1 / (1 + np.exp(-z))
axes[0].plot(z, sigmoid, color="#e74c3c", linewidth=2.5, label="σ(z) = 1/(1+e⁻ᶻ)")
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=1.2, label="Threshold = 0.5")
axes[0].axvline(0, color="gray", linestyle=":", linewidth=1.2)
axes[0].fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.15, color="#2ecc71", label="Predict Malignant")
axes[0].fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.15, color="#3498db", label="Predict Benign")
axes[0].set_title("Sigmoid Function", fontsize=13, fontweight="bold")
axes[0].set_xlabel("z (linear combination of features)")
axes[0].set_ylabel("Probability")
axes[0].legend(fontsize=9)
axes[0].set_ylim(-0.05, 1.05)

# Predicted probability distribution
axes[1].hist(y_prob[y_test == 0], bins=25, alpha=0.7, color="#3498db",
             label="Benign (actual)", edgecolor="white")
axes[1].hist(y_prob[y_test == 1], bins=25, alpha=0.7, color="#e74c3c",
             label="Malignant (actual)", edgecolor="white")
axes[1].axvline(0.5, color="black", linestyle="--", linewidth=1.5, label="Threshold = 0.5")
axes[1].axvline(0.3, color="orange", linestyle="--", linewidth=1.5, label="Medical threshold = 0.3")
axes[1].set_title("Predicted Probability Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("P(Malignant)")
axes[1].set_ylabel("Count")
axes[1].legend(fontsize=9)

plt.suptitle("Sigmoid Function & Probability Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("clf_01_sigmoid_probdist.png", bbox_inches="tight")
plt.close()
print("✅ Saved: clf_01_sigmoid_probdist.png")


STEP 6: Generating All Plots
✅ Saved: clf_01_sigmoid_probdist.png


In [8]:
# ============================================================
# PLOT 2 — Confusion Matrices (0.5 & 0.3 threshold)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (thresh, y_p, title) in zip(axes, [
    (0.5, y_pred,        "Confusion Matrix\n(Threshold = 0.5)"),
    (0.3, y_pred_medical,"Confusion Matrix\n(Medical Threshold = 0.3)")
]):
    cm_plot = confusion_matrix(y_test, y_p)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_plot,
                                  display_labels=["Benign", "Malignant"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontsize=12, fontweight="bold")

plt.suptitle("Confusion Matrices — Threshold Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("clf_02_confusion_matrices.png", bbox_inches="tight")
plt.close()
print("✅ Saved: clf_02_confusion_matrices.png")


✅ Saved: clf_02_confusion_matrices.png


In [9]:
# ============================================================
# PLOT 3 — ROC Curve
# ============================================================
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="#e74c3c", linewidth=2.5,
         label=f"Logistic Regression (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1.5, label="Random Classifier")
plt.fill_between(fpr, tpr, alpha=0.1, color="#e74c3c")

# Mark threshold 0.5 and 0.3 on ROC
for thresh_mark, color_mark, label_mark in [(0.5, "blue", "t=0.5"), (0.3, "orange", "t=0.3")]:
    idx = np.argmin(np.abs(roc_thresholds - thresh_mark))
    plt.scatter(fpr[idx], tpr[idx], s=100, color=color_mark,
                zorder=5, label=f"Threshold {label_mark}")

plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Recall / Sensitivity)")
plt.title("ROC Curve — Logistic Regression", fontsize=13, fontweight="bold")
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("clf_03_roc_curve.png", bbox_inches="tight")
plt.close()
print("✅ Saved: clf_03_roc_curve.png")

✅ Saved: clf_03_roc_curve.png


In [10]:
# ============================================================
# PLOT 4 — Precision-Recall Curve
# ============================================================
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(rec_curve, prec_curve, color="#9b59b6", linewidth=2.5,
         label=f"Logistic Regression (AP = {avg_prec:.4f})")
plt.axhline(y_test.mean(), color="gray", linestyle="--", linewidth=1.5,
            label=f"Baseline (prevalence = {y_test.mean():.2f})")
plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve", fontsize=13, fontweight="bold")
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("clf_04_precision_recall.png", bbox_inches="tight")
plt.close()
print("✅ Saved: clf_04_precision_recall.png")

✅ Saved: clf_04_precision_recall.png


In [11]:
# ============================================================
# PLOT 5 — Threshold Tuning: Precision vs Recall vs F1
# ============================================================
plt.figure(figsize=(10, 6))
plt.plot(thresh_df["Threshold"], thresh_df["Precision"], "o-", color="#3498db",
         linewidth=2, label="Precision")
plt.plot(thresh_df["Threshold"], thresh_df["Recall"],    "s-", color="#e74c3c",
         linewidth=2, label="Recall")
plt.plot(thresh_df["Threshold"], thresh_df["F1"],        "^-", color="#2ecc71",
         linewidth=2, label="F1 Score")
plt.plot(thresh_df["Threshold"], thresh_df["Accuracy"],  "D-", color="#9b59b6",
         linewidth=2, label="Accuracy")
plt.axvline(0.5, color="gray",   linestyle="--", linewidth=1.2, label="Default t=0.5")
plt.axvline(0.3, color="orange", linestyle="--", linewidth=1.2, label="Medical t=0.3")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning — Precision / Recall / F1 / Accuracy",
          fontsize=13, fontweight="bold")
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig("clf_05_threshold_tuning.png", bbox_inches="tight")
plt.close()
print("✅ Saved: clf_05_threshold_tuning.png")


✅ Saved: clf_05_threshold_tuning.png


In [12]:
# ============================================================
# PLOT 6 — Top Feature Coefficients
# ============================================================
coef_df = pd.DataFrame({
    "Feature":     X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False).head(15)

plt.figure(figsize=(10, 7))
colors = ["#e74c3c" if c > 0 else "#3498db" for c in coef_df["Coefficient"]]
bars = plt.barh(coef_df["Feature"], coef_df["Coefficient"],
                color=colors, edgecolor="white", alpha=0.85)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Top 15 Feature Coefficients\n(Red = increases Malignant probability, Blue = decreases)",
          fontsize=12, fontweight="bold")
plt.xlabel("Coefficient (Standardized)")
plt.tight_layout()
plt.savefig("clf_06_feature_coefficients.png", bbox_inches="tight")
plt.close()
print("✅ Saved: clf_06_feature_coefficients.png")

✅ Saved: clf_06_feature_coefficients.png


In [13]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"""
┌──────────────────────────────────────────────────┐
│            MODEL PERFORMANCE SUMMARY             │
├─────────────────────┬────────────────────────────┤
│ Metric              │ Value                      │
├─────────────────────┼────────────────────────────┤
│ Accuracy            │ {acc*100:.2f}%                     │
│ ROC-AUC             │ {roc_auc:.4f}                    │
│ Precision (M)       │ {precision:.4f}                    │
│ Recall (M)          │ {recall:.4f}                    │
│ F1 Score (M)        │ {f1:.4f}                    │
│ True Positives      │ {tp:<4}  (Cancer caught ✅)   │
│ False Negatives     │ {fn:<4}  (Cancer MISSED ❌)   │
│ False Positives     │ {fp:<4}  (False alarm ⚠️)     │
│ True Negatives      │ {tn:<4}  (Correctly benign ✅) │
└─────────────────────┴────────────────────────────┘
""")
print(f"  📌 Top feature: '{coef_df.iloc[0]['Feature']}' (coef={coef_df.iloc[0]['Coefficient']:.4f})")
print(f"  📌 At medical threshold 0.3: FN drops from {fn} → {fn_m} (fewer missed cancers)")
print("\n🎉 Task 4 Complete! All 6 plots saved.")



FINAL SUMMARY

┌──────────────────────────────────────────────────┐
│            MODEL PERFORMANCE SUMMARY             │
├─────────────────────┬────────────────────────────┤
│ Metric              │ Value                      │
├─────────────────────┼────────────────────────────┤
│ Accuracy            │ 96.49%                     │
│ ROC-AUC             │ 0.9960                    │
│ Precision (M)       │ 0.9750                    │
│ Recall (M)          │ 0.9286                    │
│ F1 Score (M)        │ 0.9512                    │
│ True Positives      │ 39    (Cancer caught ✅)   │
│ False Negatives     │ 3     (Cancer MISSED ❌)   │
│ False Positives     │ 1     (False alarm ⚠️)     │
│ True Negatives      │ 71    (Correctly benign ✅) │
└─────────────────────┴────────────────────────────┘

  📌 Top feature: 'texture_worst' (coef=1.4341)
  📌 At medical threshold 0.3: FN drops from 3 → 1 (fewer missed cancers)

🎉 Task 4 Complete! All 6 plots saved.
